In [11]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity #find similar songs

In [12]:
raw_df = pd.read_csv("../data/spotify_tracks.csv")
clustered_df = pd.read_csv("../data/spotify_tracks_clustered_moods.csv")

print(raw_df.shape)
print(clustered_df.shape)

(114000, 21)
(114000, 17)


In [13]:
raw_df["cluster"] = clustered_df["cluster"]
raw_df["mood_name"] = clustered_df["mood_name"]

df = raw_df
df.head()

,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre,cluster,mood_name
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic,5,Party Beast
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic,1,Acoustic Chill
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic,1,Acoustic Chill
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic,1,Acoustic Chill
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic,1,Acoustic Chill


In [14]:
feature_cols = [
    "danceability",
    "energy",
    "valence",
    "acousticness",
    "instrumentalness",
    "speechiness",
    "liveness",
    "tempo"
]

In [15]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[feature_cols])


In [ ]:
# #TO BUILD SIMILARITY SYSTEM WE HAVE 2 OPTIONS:
# 1. COSINE : Measures similarity based on angle (pattern similarity).✅
# 2. EUCLIDEAN


In [17]:
print(df.columns)

Index(['Unnamed: 0', 'track_id', 'artists', 'album_name', 'track_name',
       'popularity', 'duration_ms', 'explicit', 'danceability', 'energy',
       'key', 'loudness', 'mode', 'speechiness', 'acousticness',
       'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature',
       'track_genre', 'cluster', 'mood_name'],
      dtype='str')


In [18]:
# FIRST WE WILL Build Recommendation Function
def recommend_similar_songs(song_name, top_n=5):
    # find matching song
    matches = df[df["track_name"].str.lower() == song_name.lower()]

    if matches.empty:
        return "Song not found. Please check spelling."

    song_index = matches.index[0]

    # get cluster of that song
    song_cluster = df.loc[song_index, "cluster"]

    # filter songs in same cluster
    cluster_indices = df[df["cluster"] == song_cluster].index

    # extract vectors for cluster songs
    cluster_vectors = X_scaled[cluster_indices]

    # extract vector of input song
    song_vector = X_scaled[song_index].reshape(1, -1)

    # cosine similarity
    similarities = cosine_similarity(song_vector, cluster_vectors)[0]

    # sort indices by similarity (descending)
    sorted_idx = np.argsort(similarities)[::-1]

    # get top song indices
    top_song_indices = cluster_indices[sorted_idx]

    # remove input song itself
    top_song_indices = top_song_indices[top_song_indices != song_index]

    # select top N recommendations
    top_song_indices = top_song_indices[:top_n]

    return df.loc[top_song_indices, ["track_name", "artists", "mood_name", "cluster", "popularity"]]

In [19]:
# TO CHECK
recommend_similar_songs("Blinding Lights", top_n=5)

,track_name,artists,mood_name,cluster,popularity
38541,Mob Rule,Bad//Dreems,Party Beast,5,29
69744,Vaanampaadiyin,Sujatha,Party Beast,5,22
88712,(I Can't Help) Falling In Love With You,UB40,Party Beast,5,70
33591,Atrakce,PAWLIE POIZN;Medooza,Party Beast,5,36
100553,Me Voy,Los Victorios,Party Beast,5,34


# ##TILL HERE IT WAS RECOMMENDATION SYSTEM, NOW WE'RE GONNA BUILD MOOD BASED RECOMMENDATION SYSTEM
- ✅ Build recommendation system
    Function idea:

    input: song name
    output: top 5 similar songs from same cluster
    (using cosine similarity / euclidean distance)

- ✅ Add a “mood-based recommendation”
    Example:
    Input mood: chill → recommend 10 chill songs

In [24]:
def recommend_by_mood(mood, n=11):
    mood_matches = df[df["mood_name"].str.lower() == mood.lower()]

    if mood_matches.empty:
        return "Mood not found. Please check spelling."

    return mood_matches.sample(n=min(n, len(mood_matches)), random_state=42)[
        ["track_name", "artists", "mood_name", "cluster", "popularity"]
    ]

In [25]:
recommend_by_mood("Acoustic Chill", n=11)

,track_name,artists,mood_name,cluster,popularity
76921,"Ailein Duinn (Theme From The Movie ""Rob Roy"")",Méav,Acoustic Chill,1,20
11612,Daydreamer,Adele,Acoustic Chill,1,51
79746,I Want You To Love Me,Fiona Apple,Acoustic Chill,1,59
12360,難得有情人,Shirley Kwan,Acoustic Chill,1,47
51714,"Kanne Kanne (From ""Ayogya"")",Anirudh Ravichander;Sam C.S.,Acoustic Chill,1,57
106229,Piano in the Sky,Winona Oak,Acoustic Chill,1,0
102087,Until I Found You - Em Beihold Version,Stephen Sanchez;Em Beihold,Acoustic Chill,1,2
4530,October Birds,Flower Face,Acoustic Chill,1,43
99311,Main Toh Chala,Vismay Patel,Acoustic Chill,1,42
52771,I Saw The Light,Hank Williams;Drifting Cowboys,Acoustic Chill,1,11
